In [1]:
import time
from stable_baselines3 import PPO

from spotmicro.env.spotmicro_env import SpotmicroEnv
from spotmicro.physics.factory import create_backend
from spotmicro.devices.fixed_controller import FixedController
from spotmicro.tools.config import Config
from reward_function import reward_function, RewardState

pybullet build time: Apr  4 2025 18:56:19


In [ ]:
from stable_baselines3.common.callbacks import CheckpointCallback
from stable_baselines3.common.env_checker import check_env
from stable_baselines3.common.logger import configure

# ========= CONFIG ==========
TOTAL_STEPS = 2_000_000
run = "standPB"
log_dir = f"./logs/{run}"

def clipped_linear_schedule(initial_value, min_value=1e-5):
    def schedule(progress_remaining):
        return max(progress_remaining * initial_value, min_value)
    return schedule

checkpoint_callback = CheckpointCallback(
    save_freq=TOTAL_STEPS // 5,
    save_path=f"{run}_checkpoints",
    name_prefix=f"ppo_{run}"
)

# ========= ENV ==========
cfg = Config()
dev = FixedController("still") #not a configurable class
backend = create_backend("pybullet", use_gui=False)
env = SpotmicroEnv(
    backend,
    dev,
    cfg,
    reward_function,
    RewardState(),
    use_gui=False
)
check_env(env, warn=True)


# ========= MODEL ==========
model = PPO(
    "MlpPolicy", 
    env,
    verbose=1,   # no default printouts
    learning_rate=clipped_linear_schedule(3e-4),
    ent_coef=0.001,
    clip_range=0.1,
    tensorboard_log=log_dir,
    device = 'cpu'
)

# Custom logger: ONLY csv + tensorboard (no stdout table)
new_logger = configure(log_dir, ["csv", "tensorboard"])
model.set_logger(new_logger)

# ========= TRAIN ==========
model.learn(
    total_timesteps=TOTAL_STEPS,
    reset_num_timesteps=False,
    callback=checkpoint_callback
)
model.save(f"ppo_{run}")
env.close()

In [2]:
policy = "standPB"

cfg = Config()
dev = FixedController("still")
backend = create_backend("pybullet", use_gui=True)
env = SpotmicroEnv(
    backend,
    dev,
    cfg,
    reward_function,
    RewardState(),
    use_gui=True
)
obs, _ = env.reset()

# === Load model ===
model = PPO.load(f"ppo_{policy}", device = 'cpu')
#model = PPO.load(f"{policy}_checkpoints/ppo_{policy}_3000000_steps")
base_steps = env.num_steps

t0 = time.time()
for _ in range(3001):
    action, _ = model.predict(obs, deterministic=True)
    #action = np.array([j.from_position_to_action(hp) for j, hp in zip(env.agent.motor_joints, env.agent.homing_positions)])
    obs, reward, terminated, truncated, info = env.step(action)
    
    time.sleep(1/60.)
    print(f"Base height: {env.agent._state.base_position[2]}")
    if terminated or truncated:
        print("Terminated")
        env.plot_reward_components()  # plot per episode
        obs, _ = env.reset()
        print(f"Num steps: {env.num_steps - base_steps}")
        break
    
t1 = time.time()
print(f"Elapsed real time: {t1-t0}")

env.close()

Base height: 0.22589438622707453
Base height: 0.2257723076603863
Base height: 0.22594882161478214
Base height: 0.22611514069742492
Base height: 0.2261653485929929
Base height: 0.22595157907049163
Base height: 0.22573617175799038
Base height: 0.2257288414264294
Base height: 0.22575283809905464
Base height: 0.22576575253069048
Base height: 0.22579227681105768
Base height: 0.22578279010873553
Base height: 0.22576908632891382
Base height: 0.22574727875651301
Base height: 0.22574520656005434
Base height: 0.2257430326658802
Base height: 0.22576604357190916
Base height: 0.2257745516354265
Base height: 0.2257601051238967
Base height: 0.22575063507251822
Base height: 0.2257455489360724
Base height: 0.2257471567916113
Base height: 0.22575200894693506
Base height: 0.2257497074435651
Base height: 0.2257528284548795
Base height: 0.22576687713376123
Base height: 0.2257847516215211
Base height: 0.22577989538106272
Base height: 0.22576647262830749
Base height: 0.22575687023767788
Base height: 0.225749

error: Not connected to physics server.